# AskSight v0.6 — Two-Stage Training (Kaggle / Colab GPU)

`v0.6-b-lora16-clip-vit-large-patch14`

This notebook implements the corrected AskSight v0.6 architecture fixing all root causes identified in v0.5:
1. **No Defective Regex Rewriting:** Preserves direct short VQA answers (`yes`, `no`, `drywall`, `left`, `tennis`).
2. **Task Separation:** Short answers for direct questions, natural-language descriptions for photo captioning.
3. **Two-Stage Training:**
   - **Stage 1 (Feature Alignment):** CLIP (frozen) + Qwen2.5 (frozen). Trains only the 2-layer MLP projection connector on image-text alignment.
   - **Stage 2 (Visual Instruction Tuning):** Initializes from Stage 1 connector. Attaches LoRA rank 16 on Qwen2.5 with decoupled learning rates, warmup, and gradient accumulation.
4. **Detailed Evaluation:** Exact match, official VQA accuracy, Yes/No confusion matrix (`tp, fp, tn, fn`), and per-category breakdown.

## 0. Configuration & Hyperparameters

Adjust dataset sizes or epochs if doing a quick smoke test. Defaults align with full AskSight benchmark training.

In [ ]:
VERSION_STAGE1 = "v0.6-stage1-connector-align"
VERSION_STAGE2 = "v0.6-b-lora16-clip-vit-large-patch14"

TRAIN, VAL, TEST = 20_000, 1_000, 1_000
CAPTION_IMAGES, CAPTION_VAL_IMAGES = 2000, 200
ANSWER_STYLE = "short"  # Crucial: Preserves direct short VQA answers
SEED = 0
DTYPE = "float16"

# Stage 1 Hyperparameters (Connector alignment)
STAGE1_EPOCHS = 2
STAGE1_BATCH_SIZE = 4
STAGE1_LR = 0.001
STAGE1_GRAD_ACCUM = 4  # Effective batch size = 16

# Stage 2 Hyperparameters (Visual instruction tuning with LoRA)
STAGE2_EPOCHS = 3
STAGE2_BATCH_SIZE = 2
STAGE2_CONNECTOR_LR = 0.0005
STAGE2_LORA_LR = 0.00005  # 10x smaller learning rate for LoRA
STAGE2_GRAD_ACCUM = 8     # Effective batch size = 16
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Backbones
VISION_MODEL = "openai/clip-vit-large-patch14"
VISION_REVISION = "e073d58e5fea2f33477c3561de63fc5da34697ef"
LM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LM_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"


## 1. Setup & Environment

Clone repository and install AskSight model dependencies with LoRA support.

In [ ]:
import os
import subprocess
import sys
import importlib.metadata
from packaging.version import Version

REPO = "https://github.com/Pranjal6955/AskSight.git"
BRANCH = "dev"
WORKSPACE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_DIR = os.path.join(WORKSPACE, "AskSight")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, check=True)

%cd {REPO_DIR}/model
!pip install -q -e ".[dev,vlm_eval,lora]"

# Handle Kaggle torchao compatibility with PEFT if needed
try:
    torchao_version = Version(importlib.metadata.version("torchao"))
    if torchao_version < Version("0.16.0"):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
except importlib.metadata.PackageNotFoundError:
    pass

import torch
print("PyTorch:", torch.__version__, "| CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Please enable a GPU accelerator in notebook settings!"


## 2. Prepare Training and Validation Data

1. Materializes disjoint VQA v2 splits with **original short answers** (preserving direct ground truths).
2. Adds multi-sentence COCO image captions for scene description requests (`Describe this photo.`).

In [ ]:
%%time
# Step 1: Download VQA data with short answers
!python -m asksight_model.data \
    --out data/vqa \
    --train {TRAIN} \
    --val {VAL} \
    --test {TEST} \
    --seed {SEED} \
    --answer-style {ANSWER_STYLE}

# Step 2: Add captioning samples for scene descriptions
!python -m asksight_model.coco_captions \
    --data-dir data/vqa \
    --out data/asksight_coco_captions.jsonl \
    --val-out data/asksight_coco_captions_val.jsonl \
    --num-images {CAPTION_IMAGES} \
    --val-images {CAPTION_VAL_IMAGES} \
    --sentences-per-image 1 \
    --seed {SEED}

# Step 3: Merge captions into train.jsonl and val.jsonl
import json
from pathlib import Path

data_dir = Path("data/vqa")
train_path = data_dir / "train.jsonl"
val_path = data_dir / "val.jsonl"

train_records = [json.loads(line) for line in train_path.read_text().splitlines() if line.strip()]
val_records = [json.loads(line) for line in val_path.read_text().splitlines() if line.strip()]
held_out_ids = {str(r["image_id"]) for r in val_records} | {str(r["image_id"]) for r in [json.loads(l) for l in (data_dir / "test.jsonl").read_text().splitlines() if l.strip()]}

cap_train = [json.loads(l) for l in Path("data/asksight_coco_captions.jsonl").read_text().splitlines() if l.strip()]
cap_val = [json.loads(l) for l in Path("data/asksight_coco_captions_val.jsonl").read_text().splitlines() if l.strip()]

for r in cap_train:
    if str(r["image_id"]) not in held_out_ids:
        r["source"] = "asksight_coco_caption_train"
        train_records.append(r)

for r in cap_val:
    r["source"] = "asksight_coco_caption_val"
    val_records.append(r)

train_path.write_text("".join(json.dumps(r) + "\n" for r in train_records))
val_path.write_text("".join(json.dumps(r) + "\n" for r in val_records))
print(f"Prepared: {len(train_records)} train rows, {len(val_records)} val rows (Answer style: {ANSWER_STYLE})")


## 3. Stage 1: Train Connector Alignment (Frozen Vision & LM)

In Stage 1, Qwen2.5 and CLIP ViT-Large/14 remain **completely frozen**. Only the 2-layer MLP projection connector learns to project $257$ visual tokens into the language model's embedding space.

In [ ]:
%%time
import yaml
from pathlib import Path

with open("configs/v0.6-stage1.yaml") as f:
    s1_cfg = yaml.safe_load(f)

s1_cfg["train"]["epochs"] = STAGE1_EPOCHS
s1_cfg["train"]["batch_size"] = STAGE1_BATCH_SIZE
s1_cfg["train"]["lr"] = STAGE1_LR
s1_cfg["train"]["gradient_accumulation_steps"] = STAGE1_GRAD_ACCUM
s1_cfg["train"]["dtype"] = DTYPE
s1_cfg["train"]["version"] = VERSION_STAGE1
s1_cfg["model"]["version"] = VERSION_STAGE1

Path("configs/run_v0.6_stage1.yaml").write_text(yaml.safe_dump(s1_cfg, sort_keys=False))

!rm -rf checkpoints/{VERSION_STAGE1}
!python -m asksight_model.train \
    --config configs/run_v0.6_stage1.yaml \
    --version {VERSION_STAGE1} \
    --stage 1


## 4. Stage 2: Visual Instruction Tuning (Connector + LoRA)

In Stage 2, the pre-aligned connector weights from Stage 1 are loaded. LoRA rank 16 is attached to Qwen2.5. Training runs with decoupled learning rates ($5 \times 10^{-4}$ for connector, $5 \times 10^{-5}$ for LoRA) and warmup.

In [ ]:
%%time
with open("configs/v0.6-stage2.yaml") as f:
    s2_cfg = yaml.safe_load(f)

s2_cfg["train"]["epochs"] = STAGE2_EPOCHS
s2_cfg["train"]["batch_size"] = STAGE2_BATCH_SIZE
s2_cfg["train"]["lr"] = STAGE2_CONNECTOR_LR
s2_cfg["train"]["lora_lr"] = STAGE2_LORA_LR
s2_cfg["train"]["gradient_accumulation_steps"] = STAGE2_GRAD_ACCUM
s2_cfg["train"]["dtype"] = DTYPE
s2_cfg["train"]["version"] = VERSION_STAGE2
s2_cfg["train"]["init_projector_from"] = f"checkpoints/{VERSION_STAGE1}"
s2_cfg["model"]["version"] = VERSION_STAGE2

Path("configs/run_v0.6_stage2.yaml").write_text(yaml.safe_dump(s2_cfg, sort_keys=False))

!rm -rf checkpoints/{VERSION_STAGE2}
!python -m asksight_model.train \
    --config configs/run_v0.6_stage2.yaml \
    --version {VERSION_STAGE2} \
    --init-projector-from checkpoints/{VERSION_STAGE1} \
    --stage 2


## 5. Sample Inference & Sanity Check

Inspect predictions on 10 random held-out test questions to verify answers are concise, grounded, and free of degenerate sentence templates.

In [ ]:
import json
import random
from pathlib import Path
from PIL import Image
from asksight_model.service import LocalInferenceService

checkpoint = Path(f"checkpoints/{VERSION_STAGE2}")
assert (checkpoint / "config.yaml").is_file(), f"Checkpoint {checkpoint} missing"

records = [json.loads(l) for l in (Path("data/vqa") / "test.jsonl").read_text().splitlines() if l.strip()]
records = [r for r in records if "answers" in r]

service = LocalInferenceService(checkpoint, dtype=DTYPE)
print(f"Loaded {service.model_version} on {service.model.device.type}\n")

random.seed(SEED)
for row in random.sample(records, 10):
    img = Image.open(Path("data/vqa") / row["image"]).convert("RGB")
    pred = service.answer(img, row["question"])
    print("Q:", row["question"])
    print("Prediction:", pred)
    print("Ground Truth:", row.get("short_answer") or row["answers"][0])
    print("-" * 60)


## 6. Full Quantitative Evaluation & Confusion Matrix

Computes official VQA accuracy, normalized exact match, Yes/No confusion matrix, and category breakdowns.

In [ ]:
%%time
!python -m asksight_model.evaluate \
    --checkpoint checkpoints/{VERSION_STAGE2} \
    --split test \
    --data-dir data/vqa \
    --dtype {DTYPE}

!python -m asksight_model.evaluate \
    --checkpoint checkpoints/{VERSION_STAGE2} \
    --questions-file evaluations/asksight_v1.jsonl \
    --dtype {DTYPE}

metrics = json.loads(Path(f"checkpoints/{VERSION_STAGE2}/metrics.json").read_text())
ev = metrics.get("eval_test", {})
print("\n=================== EVALUATION RESULTS ===================")
print(f"Normalized Exact Match:        {ev.get('normalized_exact_match', 0.0):.1%}")
print(f"Official VQA Accuracy:         {ev.get('vqa_accuracy', 0.0):.1%}")
print(f"Yes/No Accuracy:               {ev.get('yes_no_accuracy', 0.0):.1%}")
print(f"Yes/No Confusion Matrix:       {ev.get('yes_no_confusion_matrix')}")
print(f"Generic Answer Frequency:      {ev.get('generic_answer_frequency', 0.0):.1%}")
print(f"Per-Category Breakdown:        {ev.get('per_category_breakdown')}")
print(f"Latency Median:                {ev.get('latency_median_s')}s (p95: {ev.get('latency_p95_s')}s)")
print("==========================================================")


## 7. Package and Download Checkpoints

Compresses and saves both Stage 1 and Stage 2 checkpoints for download.

In [ ]:
import shutil

archive_name = f"{VERSION_STAGE2}-checkpoints.zip"
!zip -qr {archive_name} checkpoints/{VERSION_STAGE1} checkpoints/{VERSION_STAGE2}

if os.path.isdir("/kaggle/working"):
    out_path = f"/kaggle/working/{archive_name}"
    if os.path.abspath(archive_name) != os.path.abspath(out_path):
        shutil.copy2(archive_name, out_path)
    print(f"Checkpoint package ready in Kaggle Output panel: {out_path} ({os.path.getsize(out_path)/1e6:.1f} MB)")
elif os.environ.get("COLAB_RELEASE_TAG"):
    from google.colab import files
    files.download(archive_name)
else:
    print(f"Checkpoint package ready: {archive_name} ({os.path.getsize(archive_name)/1e6:.1f} MB)")
